# LSTM CPD Replication

This notebook is the final presentation and reproducibility wrapper around
frozen implementation artifacts. It loads persisted outputs, imports existing
source modules, and intentionally keeps all core methodology outside notebook cells.

## Audience, Prerequisites, and Goal

This notebook is for readers who need a top-to-bottom view of the LSTM-CPD
replication pipeline after implementation artifacts have been frozen.

Prerequisites:
- the project root contains the frozen artifacts referenced below
- the `src/lstm_cpd` package is available from the project root
- this notebook is a wrapper only; no notebook-only core logic is allowed

Goal:
- surface the full pipeline in 12 ordered sections
- keep every result traceable to an artifact path or source module

## Outline

1. Implementation Contract
2. FTMO Data Contract
3. Canonical Daily-Close Layer
4. Base Features
5. CPD Outputs
6. Dataset Assembly
7. Model and Training Setup
8. Search Results
9. Selected Model
10. Causal Inference
11. Validation Evaluation
12. Reproducibility Manifest

In [ ]:
from __future__ import annotations

import csv
import importlib
import json
import sys
from itertools import islice
from pathlib import Path


def _find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in (candidate, *candidate.parents):
        if (root / "src" / "lstm_cpd").exists() and (
            root / "spec_lstm_cpd_model_revised_sole_authority.md"
        ).exists():
            return root
    raise FileNotFoundError(
        "Unable to locate the project root from the notebook execution context."
    )


PROJECT_ROOT = _find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))


def _resolve_project_path(reference: str) -> Path:
    candidate = Path(reference)
    if candidate.is_absolute():
        return candidate
    return PROJECT_ROOT / candidate


def _preview_text(path: Path, *, max_lines: int = 8) -> str:
    with path.open("r", encoding="utf-8") as handle:
        lines = [line.rstrip("\n") for line in islice(handle, max_lines)]
    return "\n".join(lines)


def _preview_csv(path: Path, *, max_lines: int = 6) -> str:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return "".join(list(islice(handle, max_lines))).rstrip()


def _preview_json(path: Path) -> str:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(payload, dict):
        preview = {key: payload[key] for key in list(payload)[:5]}
    elif isinstance(payload, list):
        preview = payload[:2]
    else:
        preview = payload
    return json.dumps(preview, indent=2)


def preview_artifact(reference: str) -> str:
    path = _resolve_project_path(reference)
    suffix = path.suffix.lower()
    if suffix == ".json":
        return _preview_json(path)
    if suffix == ".csv":
        return _preview_csv(path)
    if suffix in {".md", ".txt", ".py"}:
        return _preview_text(path)
    size_bytes = path.stat().st_size
    return f"{path.name} ({size_bytes} bytes)"


def import_module_refs(module_refs: list[str]) -> list[tuple[str, str]]:
    resolved = []
    for module_name in module_refs:
        module = importlib.import_module(module_name)
        module_file = getattr(module, "__file__", None)
        module_path = "<built-in>" if module_file is None else str(Path(module_file).resolve())
        resolved.append((module_name, module_path))
    return resolved


def render_section(
    section_id: str,
    section_title: str,
    artifact_refs: list[str],
    module_refs: list[str],
) -> dict[str, object]:
    print(f"SECTION {section_id}: {section_title}")
    print(f"Project root: {PROJECT_ROOT}")
    print("")
    print("Artifacts")
    for artifact_ref in artifact_refs:
        print(f"- {artifact_ref}")
        print(preview_artifact(artifact_ref))
        print("")
    print("Modules")
    for module_name, module_path in import_module_refs(module_refs):
        print(f"- {module_name}: {module_path}")
    return {
        "section_id": section_id,
        "section_title": section_title,
        "artifact_refs": artifact_refs,
        "module_refs": module_refs,
    }

## 01. Implementation Contract

Freeze the authority documents that define the allowed methodology, execution-policy rules, and exclusions.

In [ ]:
SECTION_ID = 'implementation_contract'
SECTION_TITLE = 'Implementation Contract'
ARTIFACT_REFS = ['spec_lstm_cpd_model_revised_sole_authority.md', 'lstm_cpd_implementation_plan.md', 'docs/contracts/invariant_ledger.md', 'docs/contracts/exclusions_ledger.md', 'docs/contracts/execution_policy_rules.md']
MODULE_REFS = []

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 02. FTMO Data Contract

Show the admissible FTMO universe, D-timeframe path resolution, and the schema and screening reports that bound the raw input layer.

In [ ]:
SECTION_ID = 'ftmo_data_contract'
SECTION_TITLE = 'FTMO Data Contract'
ARTIFACT_REFS = ['artifacts/manifests/ftmo_asset_universe.json', 'artifacts/manifests/d_timeframe_path_manifest.json', 'docs/contracts/daily_close_schema_contract.md', 'artifacts/reports/schema_inspection_report.csv', 'artifacts/reports/asset_eligibility_report.csv', 'artifacts/reports/asset_exclusion_report.csv', 'artifacts/reports/minimum_history_screening_report.csv']
MODULE_REFS = ['lstm_cpd.ftmo_asset_universe', 'lstm_cpd.daily_close_contract', 'lstm_cpd.raw_history_screening']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 03. Canonical Daily-Close Layer

Reference the frozen canonical close manifest that anchors all later feature and sequence work.

In [ ]:
SECTION_ID = 'canonical_daily_close_layer'
SECTION_TITLE = 'Canonical Daily-Close Layer'
ARTIFACT_REFS = ['artifacts/manifests/canonical_daily_close_manifest.json']
MODULE_REFS = ['lstm_cpd.canonical_daily_close_store']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 04. Base Features

Summarize the deterministic return, volatility, normalized-return, MACD, and winsorization stack that produces the eight non-CPD features.

In [ ]:
SECTION_ID = 'base_features'
SECTION_TITLE = 'Base Features'
ARTIFACT_REFS = ['artifacts/reports/feature_provenance_report.md']
MODULE_REFS = ['lstm_cpd.features.returns', 'lstm_cpd.features.volatility', 'lstm_cpd.features.normalized_returns', 'lstm_cpd.features.macd', 'lstm_cpd.features.winsorize']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 05. CPD Outputs

Tie the GP-based CPD engine to its progress, telemetry, failure, fallback, and manifest artifacts.

In [ ]:
SECTION_ID = 'cpd_outputs'
SECTION_TITLE = 'CPD Outputs'
ARTIFACT_REFS = ['docs/contracts/cpd_engine_contract.md', 'artifacts/reports/cpd_precompute_progress.csv', 'artifacts/reports/cpd_fit_telemetry.csv', 'artifacts/reports/cpd_failure_ledger.csv', 'artifacts/reports/cpd_fallback_ledger.csv', 'artifacts/manifests/cpd_feature_store_manifest.json']
MODULE_REFS = ['lstm_cpd.cpd.gp_kernels', 'lstm_cpd.cpd.fit_window', 'lstm_cpd.cpd.precompute', 'lstm_cpd.cpd.telemetry']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 06. Dataset Assembly

Reference the frozen dataset registry and the source modules that create joins, chronological splits, sequences, and array-ready registries.

In [ ]:
SECTION_ID = 'dataset_assembly'
SECTION_TITLE = 'Dataset Assembly'
ARTIFACT_REFS = ['artifacts/manifests/dataset_registry.json']
MODULE_REFS = ['lstm_cpd.datasets.join_and_split', 'lstm_cpd.datasets.sequences', 'lstm_cpd.datasets.registry']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 07. Model and Training Setup

Expose the shared LSTM runtime, Sharpe-loss contract, single-candidate runner, and the smoke-train fidelity artifacts.

In [ ]:
SECTION_ID = 'model_training_setup'
SECTION_TITLE = 'Model and Training Setup'
ARTIFACT_REFS = ['docs/contracts/model_runtime_contract.md', 'artifacts/training/training_runner_contract.md', 'artifacts/training/smoke_run/smoke_config.json', 'artifacts/training/smoke_run/smoke_best_model.keras', 'artifacts/training/smoke_run/smoke_epoch_log.csv', 'artifacts/training/smoke_run/smoke_validation_history.csv', 'artifacts/reports/model_fidelity_report.md']
MODULE_REFS = ['lstm_cpd.model.network', 'lstm_cpd.training.losses', 'lstm_cpd.training.train_candidate']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 08. Search Results

Display the immutable search schedule and the search summary report that collapses all candidate outcomes into a reviewable table.

In [ ]:
SECTION_ID = 'search_results'
SECTION_TITLE = 'Search Results'
ARTIFACT_REFS = ['artifacts/training/search_schedule.json', 'artifacts/training/search_schedule.csv', 'artifacts/reports/search_summary_report.csv']
MODULE_REFS = ['lstm_cpd.training.search_schedule', 'lstm_cpd.training.search_runner']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 09. Selected Model

Show the winning-candidate metadata and the source helpers that resolve the selected checkpoint and candidate configuration.

In [ ]:
SECTION_ID = 'selected_model'
SECTION_TITLE = 'Selected Model'
ARTIFACT_REFS = ['artifacts/training/best_candidate.json', 'artifacts/training/best_config.json']
MODULE_REFS = ['lstm_cpd.model_source', 'lstm_cpd.training.selection']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 10. Causal Inference

Reference the latest causal sequence manifest and the next-day position outputs from the online inference path.

In [ ]:
SECTION_ID = 'causal_inference'
SECTION_TITLE = 'Causal Inference'
ARTIFACT_REFS = ['artifacts/inference/latest_positions.csv', 'artifacts/inference/latest_sequence_manifest.csv']
MODULE_REFS = ['lstm_cpd.inference.online_inference']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 11. Validation Evaluation

Show the raw and rescaled validation outputs and the report that explains the model-faithful FTMO-vs.-paper evaluation boundary.

In [ ]:
SECTION_ID = 'validation_evaluation'
SECTION_TITLE = 'Validation Evaluation'
ARTIFACT_REFS = ['artifacts/evaluation/raw_validation_returns.csv', 'artifacts/evaluation/raw_validation_metrics.json', 'artifacts/evaluation/rescaled_validation_returns.csv', 'artifacts/evaluation/rescaled_validation_metrics.json', 'artifacts/evaluation/evaluation_report.md']
MODULE_REFS = ['lstm_cpd.evaluation.validation_evaluation']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

## 12. Reproducibility Manifest

Close the notebook with the manifest that hashes the selected model inputs and binds the frozen evaluation run back to the repo.

In [ ]:
SECTION_ID = 'reproducibility_manifest'
SECTION_TITLE = 'Reproducibility Manifest'
ARTIFACT_REFS = ['artifacts/reproducibility/reproducibility_manifest.json']
MODULE_REFS = ['lstm_cpd.reproducibility.manifest']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)